## Runnables 
The Foundational Building Blocks of modern Langchain

In Langchain, we have many components - LLMs, Prompts, Retrievers, Doc Loaders, Vectorbases, Parsers, etc...   
Initially these all components were developed separately, and hence had different interface and functions like .predict() for llm, .format() for prompt.     
To connect these components, Langchain developed "Chains", different chain for each task. For example- to connect llm and prompt we had LlmChain(), and similarly for every connection we got different chain. This was too much to handle since there are so many type of applications that can be built and they may require many type of chains. So langchain built so many chains - Their Code base got huge and learning curve of langchain got harder. 


To solve this problem, they introduced: Runnables     
So now every component in langchain is a Runnable. 

Runnables are foundational building blocks of modern langchain - 
- Every runnable do three things
    - Take input -> Process -> Give output
- Every runnable have some standard methods to give them common interface like
    - .invoke() , .batch(), .stream()
- Composability: Since each runnable have common interface they can be chained easily through -> "|" 
    - Each component can be chained easily. And since a chain is also a runnable, multiple chains can also be chained together.



How is this implemented?   
Basically, they have created an abstract class "Runnables" and made all the components inherit from that class, so each component have common methods.

There are two types of Runnables
1. ___Task Specific Runnable___
    - These are components of langchain that have to converted into runnables so they integrate seamlessly in the pipeline
    - These perform task specific operations like: llm calls, prompting, retrieval etc
    - Ex: ChatHuggingFace, ChatOpenAI, PromptTemplate, ChatPromptTemplate, StrOutputParser etc 
    

2. ___Runnable Primitives___
    - These are fundamental building blocks of logic in AI pipelines
    - These help orchestrate execution of task specific runnables by defining how different runnables interact
    - Ex: 
        - `RunnableSequence`: Run steps in order ("|" operator)
        - `RunnableParallel`: Takes input, and pass that to multiple parallel chains
        - `RunnableBranch`: 
        - `RunnableMap`:
        - `RunnableLambda`: This allows you to apply custom python functions withing AI pipeline. This acts as middleware between difference AI components, which enables transformation, filtering, preproccessing in between AI workflow
        - `RunnablePassthrough`:Takes input, does no changes, and return that input as output

In [27]:
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.runnables import RunnableParallel, RunnableBranch, RunnableLambda, RunnableMap, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from dotenv import load_dotenv

In [28]:
load_dotenv()

True

In [29]:
llm = HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-V4-Flash",
    task='text-generation',
    temperature=0.8
)

model = ChatHuggingFace(llm=llm)

In [32]:
parser = StrOutputParser()

def is_large(text):
    return len(text.split()) > 100 

template1 = PromptTemplate(
    template="Explain this topic to me like i am a five year old \n Topic: {topic}",
    input_variables=['topic'])

template2 = PromptTemplate(
    template="Generate 5 basic question quiz on this topic: {topic}",
    input_variables=['topic'])

template3 = PromptTemplate(
    template="Summarize the notes and give answers of quiz. \n Topic:\n{topic} \n\n Notes:\n{notes} \n\n Quiz:\n{quiz}",
    input_variables=['topic', 'notes','quiz'])

template4 = PromptTemplate(
    template="Compress the text into 100 or less words: Text {Summary}",
    input_variables=['Summary']
)

notes_quiz = RunnableParallel(
    {'notes': template1 | model | parser,
     'quiz': template2 | model | parser,
     'topic':RunnablePassthrough()})        #Whatever is the input to this chain, will be stored in 'topic', hence topic is stored

summary = RunnableParallel(
    {'Summary': template3 | model | parser,
     'NQT': RunnablePassthrough()})        #Notes, Quiz, Topic is stored in this key)


runnable_branch_for_wc = RunnableBranch(
    (lambda x:is_large(x['Summary']), 
        RunnableParallel({
            'original_word_count': RunnableLambda(lambda x: len(x['Summary'].split())),
            'Compressed_Summary': template4 | model | parser,
            'SNQT':RunnablePassthrough()})), 
    RunnableParallel({
        'original_word_count': RunnableLambda(lambda x: len(x['Summary'].split())),
        'SNQT':RunnablePassthrough()})                          #Default
)
final_chain = notes_quiz | summary | runnable_branch_for_wc

In [33]:
result = final_chain.invoke({'topic':'Support Vector Machines'})

In [64]:
print("Type of result is:",type(result))
print("Keys:",result.keys())
print(type(result['original_word_count']))
print(type(result['Compressed_Summary']))
print(type(result['SNQT']))

print("_"*40, "\n")
print(result['SNQT'].keys())
print(type(result['SNQT']['Summary']))
print(type(result['SNQT']['NQT']))

print("_"*40, "\n")
print(result['SNQT']['NQT'].keys())
print(type(result['SNQT']['NQT']['notes']))
print(type(result['SNQT']['NQT']['quiz']))
print(type(result['SNQT']['NQT']['topic']))

print("_"*40, "\n")
print(result['SNQT']['NQT']['topic'].keys())
print(type(result['SNQT']['NQT']['topic']['topic']))

print("Hence, we have kept all - Topic, Notes, Quiz, Summary, Compressed_Summary, wordcount ")

Type of result is: <class 'dict'>
Keys: dict_keys(['original_word_count', 'Compressed_Summary', 'SNQT'])
<class 'int'>
<class 'str'>
<class 'dict'>
________________________________________ 

dict_keys(['Summary', 'NQT'])
<class 'str'>
<class 'dict'>
________________________________________ 

dict_keys(['notes', 'quiz', 'topic'])
<class 'str'>
<class 'str'>
<class 'dict'>
________________________________________ 

dict_keys(['topic'])
<class 'str'>
Hence, we have kept all - Topic, Notes, Quiz, Summary, Compressed_Summary, wordcount 


In [66]:
from IPython.display import Markdown, display
display(Markdown(result['SNQT']['NQT']['topic']['topic']))
display(Markdown(result['SNQT']['NQT']['notes']))
display(Markdown(result['SNQT']['NQT']['quiz']))
print("Original Word Count in summary below:")
print(result['original_word_count'])
display(Markdown(result['Compressed_Summary']))


Support Vector Machines

Imagine you have two big piles of your favorite toys: a pile of red toy cars and a pile of blue toy blocks. They’re all mixed up on the floor.

Your job is to draw a single, long, straight line on the floor with your crayon to separate the red cars from the blue blocks. You want all the red cars on one side of the linehare and all the blue blocks on the other side.

Now, what’s the *best* way to draw that line? You don’t want to draw it super close to the toys, because if your little brother kicks one, it might roll over the line and get mixed up again.

So, you look for the **widest, emptiest path** you

Here’s a 5-question basic quiz on Support Vector Machines (SVM), with multiple-choice answers and an answer key at the end.

---

### SVM Basics Quiz

**1. What is the primary goal of a Support Vector Machine (SVM)?**  
A. To minimize the number of support vectors  
B. To find the hyperplane that maximizes the margin between classes  
C. To reduce the dimensionality of the data  
D. To cluster data points based on similarity  

**2. In SVM, what are “support vectors”?**  
A. All data points in the training set  
B. Only the data points that lie exactly on the hyperplane  
C. The data points closest to the decision boundary that define the margin  
D. The points that are misclassified during training  

**3. What is the “kernel trick” in SVM used for?**  
A. To speed up training on small datasets  
B. To transform data into a higher-dimensional space without explicitly computing the transformation  
C. To remove outliers from the training set  
D. To reduce the number of support vectors  

**4. Which of the following best describes the “margin” in SVM?**  
A. The distance between the two classes in the original feature space  
B. The number of support vectors used to define the decision boundary  
C. The distance between the separating hyperplane and the nearest data point from either class  
D. The error rate on the training data  

**5. SVM is generally considered a:**  
A. Unsupervised learning algorithm  
B. Reinforcement learning algorithm  
C. Supervised learning algorithm  
D. Dimensionality reduction technique  

---

### Answer Key
1. **B** – The goal is to maximize the margin between classes.  
2. **C** – Support vectors are the data points closest to the hyperplane that define the margin.  
3. **B** – The kernel trick allows SVM to handle non-linear boundaries by mapping data to a higher dimension implicitly.  
4. **C** – The margin is the distance from the hyperplane to the nearest data points (

Original Word Count in summary below:
144


SVM explained with toys: separate red cars and blue blocks with a line that creates the widest, emptiest path (maximizing margin) to avoid misclassification from small disturbances. Key points: hyperplane maximizing margin (B); support vectors are closest data points (C); kernel trick transforms to higher dimensions without explicit computation (B); margin is distance to nearest point (C); SVM is supervised learning (C).